# Lesson 14 — prefilter vs postfilter · กรองก่อนหรือกรองหลัง

`where` กับ vector search ทำงานร่วมกันได้สองแบบ
**prefilter** กรองแถวด้วย `where` ก่อน แล้วค่อยหา nearest ในกลุ่มที่เหลือ
**postfilter** หา top-k จากทุกแถวก่อน แล้วค่อยกรอง ถ้า top-k ไม่มีแถวที่ตรง `where` ก็ได้ศูนย์

ข้อมูลคือโพสต์จริงของ Nat 11 โพสต์ สามหัวข้อ `memory` `agents` `hardware`
vector 3 มิติทำมือ แกนคือ `[memory, agents, hardware]`

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

import lancedb
db = lancedb.connect("./data")
tbl = db.create_table("posts", data=load("nat_posts.jsonl"), mode="overwrite")
tbl.to_pandas()[["id", "date", "topic", "vector", "text"]].assign(text=lambda d: d.text.str[:40])

[2026-09-10T11:48:41Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/14-prefilter-postfilter/data/posts.lance, it will be created


,id,date,topic,vector,text
0,p01,2026-08-28,memory,"[0.9, 0.1, 0.0]",ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้อ
1,p02,2026-08-20,memory,"[1.0, 0.0, 0.0]",หนังสือ... บันทึกการ... ค่อยๆสร้าง Vecto
2,p03,2026-08-15,memory,"[0.8, 0.2, 0.0]",ความทรงจำระหว่างบรรทัดของ Human และ Clau
3,p04,2026-07-31,memory,"[0.9, 0.0, 0.1]",ทำอินเด็กซ์ Embedding เข้าสู่ Vector Spa
4,p05,2026-06-22,memory,"[0.7, 0.3, 0.0]",Visualize ว่า เรา หรือใคร คุยกับ AI ด้วย
5,p06,2026-08-20,agents,"[0.1, 0.9, 0.0]",Claude Code ใน Messenger ครับ ทำเสร็จแล้
6,p07,2026-06-17,agents,"[0.0, 1.0, 0.0]",เข้าสู่ยุค Multi-Agents แบบเต็มตัว.... C
7,p08,2026-05-20,agents,"[0.1, 0.9, 0.0]",Multi Agent แบบ Team .... ของ Claude Cod
8,p09,2026-05-31,hardware,"[0.0, 0.3, 0.7]",ทำเฟิร์มแวร์เอา Claude Code มาออกจอเลยคร
9,p10,2026-05-30,hardware,"[0.0, 0.2, 0.8]",เอาจอ มาต่อ Claude Code BLE Bridge ลองแล


คำถาม "โพสต์เรื่อง **hardware** ที่ใกล้เรื่อง memory ที่สุด 2 โพสต์"
ทิศ memory คือ `[1, 0, 0]`
ไม่มี `where` ก่อน top-2 คือ p02 p01 ทั้งคู่เป็น memory (p01 กับ p04 ห่างเท่ากัน 0.02 เลือกตัวแรก)

In [3]:
q = [1.0, 0.0, 0.0]
tbl.search(q).limit(2).to_pandas()[["id", "topic", "_distance", "text"]].assign(text=lambda d: d.text.str[:40])

,id,topic,_distance,text
0,p02,memory,0.00,หนังสือ... บันทึกการ... ค่อยๆสร้าง Vecto
1,p01,memory,0.02,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้อ


**prefilter** — กรองเหลือ hardware 3 โพสต์ก่อน แล้วค่อยวัดระยะ
ได้ p09 p10 โพสต์จอกับเฟิร์มแวร์ ใกล้ memory ที่สุดในกลุ่ม hardware

In [4]:
tbl.search(q).where("topic = 'hardware'", prefilter=True).limit(2).to_pandas()[["id", "topic", "_distance", "text"]].assign(text=lambda d: d.text.str[:40])

,id,topic,_distance,text
0,p09,hardware,1.58,ทำเฟิร์มแวร์เอา Claude Code มาออกจอเลยคร
1,p10,hardware,1.68,เอาจอ มาต่อ Claude Code BLE Bridge ลองแล


**postfilter** — เอา top-2 จากทุกโพสต์ (p02 p01) แล้วค่อยกรองเหลือ hardware
ไม่เหลือสักแถว ตารางว่าง ทั้งที่โพสต์ hardware มีอยู่ 3 โพสต์

In [5]:
tbl.search(q).where("topic = 'hardware'", prefilter=False).limit(2).to_pandas()[["id", "topic", "_distance", "text"]]

,id,topic,_distance,text


postfilter จะเห็น hardware ก็ต่อเมื่อ `limit` ใหญ่พอให้หลุดเข้า top-k
`limit(11)` = ทุกโพสต์ ค่อยได้ครบสาม

ทำไมถึงมี postfilter ให้เลือก
กับ index (บทที่ 11) prefilter ต้องกรองก่อนเดินเข้า index บางกรณีช้ากว่า
postfilter เร็วกว่าแต่เสี่ยงหาย สำหรับความจำ agent ที่กรองด้วย session หรือ agent id ใช้ prefilter

In [6]:
tbl.search(q).where("topic = 'hardware'", prefilter=False).limit(11).to_pandas()[["id", "topic", "_distance", "text"]].assign(text=lambda d: d.text.str[:40])

,id,topic,_distance,text
0,p09,hardware,1.58,ทำเฟิร์มแวร์เอา Claude Code มาออกจอเลยคร
1,p10,hardware,1.68,เอาจอ มาต่อ Claude Code BLE Bridge ลองแล
2,p11,hardware,2.00,เตรียมตัวแปลงร่างกันครับ! รอบทความแผ้บบบ
